# Step 1 — Data Acquisition

Downloads the `adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal` dataset
from Kaggle, unzips it into `data/raw/`, and performs a quick sanity check.

**Prerequisites:**
- `kaggle.json` placed at `~/.kaggle/kaggle.json` (Linux/macOS) or `%USERPROFILE%\.kaggle\kaggle.json` (Windows).
- Run from the repo root: `airsight/`

## 1.1 — Install / verify dependencies

In [ ]:
# Upgrade kaggle silently; all other deps are stdlib
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"])
print("kaggle package ready.")

## 1.2 — Locate / validate `kaggle.json`

In [ ]:
import os, stat, platform
from pathlib import Path

# Kaggle CLI looks for credentials in KAGGLE_CONFIG_DIR or the default home location
kaggle_dir = Path(os.environ.get("KAGGLE_CONFIG_DIR", Path.home() / ".kaggle"))
kaggle_json = kaggle_dir / "kaggle.json"

if not kaggle_json.exists():
    raise FileNotFoundError(
        f"kaggle.json not found at {kaggle_json}.\n"
        "Place your API token there before continuing."
    )

# On Linux/macOS the file must be chmod 600; Windows ignores this but we set it anyway
if platform.system() != "Windows":
    kaggle_json.chmod(stat.S_IRUSR | stat.S_IWUSR)

print(f"✔  kaggle.json found at: {kaggle_json}")

## 1.3 — Create the output directory

In [ ]:
# Resolve repo root regardless of where the kernel CWD lands
REPO_ROOT = Path("__file__").resolve().parent.parent  # notebooks/ -> airsight/

# Fallback: if __file__ isn't defined (interactive kernel), anchor to the notebook location
try:
    REPO_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    REPO_ROOT = Path.cwd().parent  # assumes CWD is airsight/notebooks/

RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root  : {REPO_ROOT}")
print(f"Raw data   : {RAW_DATA_DIR}")

## 1.4 — Download dataset via Kaggle API

In [ ]:
import kaggle  # noqa: F401  (triggers credential load)
from kaggle.api.kaggle_api_extended import KaggleApiExtended

DATASET_SLUG = "adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal"

api = KaggleApiExtended()
api.authenticate()

print(f"Downloading '{DATASET_SLUG}' ...")
print(f"Destination : {RAW_DATA_DIR}")
print("This may take several minutes depending on dataset size.\n")

api.dataset_download_files(
    DATASET_SLUG,
    path=str(RAW_DATA_DIR),
    unzip=True,   # unzip in-place; removes the .zip after extraction
    quiet=False,  # show a progress bar
)

print("\n✔  Download and extraction complete.")

## 1.5 — Inspect the extracted directory structure

In [ ]:
import os

# Walk the raw directory; print a condensed tree (max 3 levels deep)
MAX_DEPTH = 3

def print_tree(root: Path, prefix: str = "", depth: int = 0):
    if depth > MAX_DEPTH:
        return
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        print(prefix + connector + entry.name)
        if entry.is_dir() and depth < MAX_DEPTH:
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(entry, prefix + extension, depth + 1)

print(f"data/raw/")
print_tree(RAW_DATA_DIR)

## 1.6 — Count files by extension (sanity check)

In [ ]:
from collections import Counter

ext_counter = Counter(
    p.suffix.lower()
    for p in RAW_DATA_DIR.rglob("*")
    if p.is_file()
)

print("File counts by extension:")
for ext, count in ext_counter.most_common():
    label = ext if ext else "(no extension)"
    print(f"  {label:15s}  {count:>6,}")

# Warn if no images were found
image_exts = {".jpg", ".jpeg", ".png", ".webp"}
total_images = sum(ext_counter[e] for e in image_exts)
print(f"\nTotal image files : {total_images:,}")

if total_images == 0:
    print("⚠  WARNING: No image files detected — verify the dataset slug and retry.")
else:
    print("✔  Image files found.")

## 1.7 — Locate CSV label files

In [ ]:
import pandas as pd

csv_files = list(RAW_DATA_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "No CSV files found under data/raw/.\n"
        "Check the dataset contents in the tree printed above."
    )

print(f"Found {len(csv_files)} CSV file(s):\n")
for csv_path in csv_files:
    df_preview = pd.read_csv(csv_path, nrows=5)
    print(f"{'─' * 60}")
    print(f"  File    : {csv_path.relative_to(REPO_ROOT)}")
    print(f"  Shape   : {pd.read_csv(csv_path).shape}")
    print(f"  Columns : {list(df_preview.columns)}")
    print()
    print(df_preview.to_string(index=False))
    print()

## 1.8 — Verify required label columns exist

In [ ]:
REQUIRED_TARGETS = ["PM2.5", "PM10", "O3", "CO", "SO2", "NO2", "AQI"]

# Check each CSV — at least ONE must contain all required columns
valid_csvs = []
for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    missing = [col for col in REQUIRED_TARGETS if col not in df.columns]
    if not missing:
        valid_csvs.append(csv_path)
        print(f"✔  {csv_path.name}  — all 7 target columns present.")
    else:
        print(f"⚠  {csv_path.name}  — missing columns: {missing}")

if not valid_csvs:
    raise ValueError(
        "None of the CSVs contain all required target columns.\n"
        f"Required: {REQUIRED_TARGETS}\n"
        "Update REQUIRED_TARGETS or check the dataset documentation."
    )

print(f"\n✔  Step 1 complete. Proceed to notebook 02_build_dataset.ipynb")